In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from module_2 import compute_ecg_features, compute_resp_features_
import pickle

%matplotlib qt

In [2]:
# ── load data ──
with open("all_device_data.pkl", "rb") as f:
    data = pickle.load(f)

In [3]:
fs=250
subjects = [i+1 for i in range(5)]
configs = ['wire', 'patch']
activity = ['laying', 'walking']


In [4]:
def segment_and_extract_respiration(ref_respiration, dev_ip, dev_gyr):
    resp_sig_len = len(ref_respiration)
    resp_window_sec = 30
    resp_step_sec=10

    resp_W = int(resp_window_sec * fs)
    resp_step = int(resp_step_sec * fs)
    resp_segments = [(s, s + resp_W) for s in range(0, resp_sig_len - resp_W + 1, resp_step)]

    final_rr = []
    resp_ref_all_segs = []

    dev_ip_peaks_all_segs = []
    dev_gyr_peaks_all_segs = []
    dev_ref_resp_peaks_all_segs = []

    spi_ip_all_segs = []
    spi_gyr_all_segs = []
    spi_ref_all_segs = []

    for i, (s, e) in enumerate(resp_segments):
        resp_ip, ip_peaks, spi_ip = compute_resp_features_(dev_ip[s:e])
        resp_gyr, gyr_peaks, spi_gyr = compute_resp_features_(dev_gyr[s:e])
        resp_ref, ref_peaks, spi_ref = compute_resp_features_(ref_respiration[s:e])
        
        resp_ref_all_segs.append(resp_ref)

        '''
        adding segment along with the extracted peaks for diagnosis purposes;
        Not returning it for now, but just need to return following three arrays
        if needed
        '''
        dev_ip_peaks_all_segs.append([dev_ip[s:e], ip_peaks])
        dev_gyr_peaks_all_segs.append([dev_gyr[s:e], gyr_peaks])
        dev_ref_resp_peaks_all_segs.append([ref_respiration[s:e], ref_peaks])
        
        spi_ip_all_segs.append(spi_ip)
        spi_gyr_all_segs.append(spi_gyr)
        spi_ref_all_segs.append(spi_ref)

        ip_nan = np.isnan(resp_ip)
        gyr_nan = np.isnan(resp_gyr)

        if ip_nan and gyr_nan:
            final_rr.append(float('nan'))
        elif ip_nan or gyr_nan:
            final_rr.append(float(resp_gyr if ip_nan else resp_ip))
        else:
            final_rr.append(float(np.mean([resp_ip, resp_gyr])))

    resp_error = []
    for idx, rr in enumerate(final_rr):
        resp_error.append(abs(rr - resp_ref_all_segs[idx]))

    return final_rr, resp_ref_all_segs, resp_error, spi_ip_all_segs, spi_gyr_all_segs, spi_ref_all_segs

In [5]:
rr_comp = pd.DataFrame(columns=['subject', 'config', 'activity', 'dev_rr', 'ref_rr', 'error', 'spi_ip', 'spi_gyr', 'spi_ref'])
# hr_comp = pd.DataFrame(columns=['dev_hr', 'ref_hr', 'error'])

for act in activity:
    for config in configs:
        for sub in subjects:
            print(f"subject_{sub}_{config}_{act}")

            try:
                dev_ip = data[f"subject_{sub}/{config}/{act}"]["respiration"]["pre_aligned_prep_dev_ip"]
                dev_gyr = data[f"subject_{sub}/{config}/{act}"]["respiration"]["pre_aligned_prep_dev_gyr"]
                ref_resp = data[f"subject_{sub}/{config}/{act}"]["respiration"]["pre_aligned_prep_ref_respiration"]

                # dev_lead2 = data[f"subject_{sub}/{config}/{act}"]["ecg"]["dev_lead2"]
                # ref_lead2 = data[f"subject_{sub}/{config}/{act}"]["ecg"]["ref_lead2"]

                final_rr, resp_ref_all_segs, resp_error, spi_ip, spi_gyr, spi_ref = segment_and_extract_respiration(ref_resp, dev_ip, dev_gyr)

                new_df = pd.DataFrame({
                    'subject': f"subject_{sub}",
                    'config': config,
                    'activity': act,
                    'dev_rr': final_rr,
                    'ref_rr': resp_ref_all_segs,
                    'error': resp_error,
                    'spi_ip': spi_ip,
                    'spi_gyr': spi_gyr,
                    'spi_ref': spi_ref
                })
                rr_comp = pd.concat([rr_comp, new_df], ignore_index=True)
            except Exception as e:
                print(f"Error: {e}")
                continue
                

subject_1_wire_laying


C:\Users\usazma\AppData\Local\Temp\ipykernel_23940\1205284576.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rr_comp = pd.concat([rr_comp, new_df], ignore_index=True)


subject_2_wire_laying
subject_3_wire_laying
subject_4_wire_laying
subject_5_wire_laying
subject_1_patch_laying
subject_2_patch_laying
subject_3_patch_laying
subject_4_patch_laying
subject_5_patch_laying
subject_1_wire_walking
subject_2_wire_walking
subject_3_wire_walking
subject_4_wire_walking
subject_5_wire_walking
subject_1_patch_walking
subject_2_patch_walking
Error: 'subject_2/patch/walking'
subject_3_patch_walking
subject_4_patch_walking
subject_5_patch_walking
Error: 'subject_5/patch/walking'


In [6]:
rr_comp.to_csv("Respiration_features.csv")

In [ ]:
config = "wire"
activity = "walking"

dev_path = f"subject_{subject}/{config}/dev/{activity}_dev.bin"
ref_ecg_path = f"subject_{subject}/{config}/reference/{activity}_ecg.EDF"
ref_resp_path = f"subject_{subject}/{config}/reference/{activity}_resp.acq"

In [ ]:
# data = read_and_clean(dev_path, ref_ecg_path, ref_resp_path)

In [ ]:
# ── load ──
with open("all_device_data.pkl", "rb") as f:
    data = pickle.load(f)

# everything comes back exactly as it was
signal = data["subject_4/wire/walking"]["respiration"]["prep_ip"]

In [ ]:

'''
Accessing data class: 

data.respiration.raw_ip       # raw
data.respiration.prep_ip      # preprocessed
data.respiration.dev_ip       # aligned

data.ecg.raw_dev_lead2        # raw
data.ecg.prep_lead2           # preprocessed
data.ecg.dev_lead2            # aligned
'''

## below are the raw signals

In [ ]:
raw_dev_lead2 = np.array(data.ecg.raw_dev_lead2)
raw_ref_lead2 = np.array(data.ecg.raw_ref_lead2)

raw_ip = np.array(data.respiration.raw_ip)
raw_gyr = np.array(data.respiration.raw_gyr)
raw_ref = np.array(data.respiration.raw_ref)

In [ ]:
plt.figure()
plt.plot(raw_dev_lead2, label="raw_dev_lead2")
plt.plot(raw_ref_lead2, label="raw_ref_lead2")
plt.title("raw_lead2_comparison")
plt.legend()
plt.show

plt.figure()
plt.plot(raw_ip, label="raw_ip")
plt.plot(raw_gyr, label="raw_gyr")
plt.plot(raw_ref, label="raw_ref")
plt.title("raw_respiration_comparison")
plt.legend()
plt.show

## below are the pre aligned preprocessed signals

In [ ]:
pre_aligned_prep_ip = np.array(data.respiration.pre_aligned_prep_dev_ip)
pre_aligned_prep_gyr = np.array(data.respiration.pre_aligned_prep_dev_gyr)
pre_aligned_prep_ref_resp = np.array(data.respiration.pre_aligned_prep_ref_respiration)

In [ ]:
plt.figure()
plt.plot(pre_aligned_prep_ip, label="pre_aligned_prep_ip")
plt.plot(pre_aligned_prep_gyr, label="pre_aligned_prep_gyr")
plt.plot(pre_aligned_prep_ref_resp, label="pre_aligned_prep_ref_resp")
plt.title("pre_aligned_preprocessed_respiration_comparison")
plt.legend()
plt.show

## below are the filtered signals

In [ ]:
prep_lead2 = np.array(data.ecg.prep_lead2)
prep_ref_ecg = np.array(data.ecg.prep_ref_ecg)

prep_ip = np.array(data.respiration.prep_ip)
prep_gyr = np.array(data.respiration.prep_gyr)
prep_ref_resp = np.array(data.respiration.prep_ref)

In [ ]:
plt.figure()
plt.plot(prep_lead2, label="prep_lead2")
plt.plot(prep_ref_ecg, label="prep_ref_ecg")
plt.title("filtered_lead2_comparison")
plt.legend()
plt.show

plt.figure()
plt.plot(prep_ip, label="prep_ip")
plt.plot(prep_gyr, label="prep_gyr")
plt.plot(prep_ref_resp, label="prep_ref_resp")
plt.title("filtered_respiration_comparison")
plt.legend()
plt.show

## below are the aligned signals

In [ ]:
dev_lead2 = np.array(data.ecg.dev_lead2)
ref_lead2 = np.array(data.ecg.ref_lead2)

dev_ip = np.array(data.respiration.dev_ip)
dev_gyr = np.array(data.respiration.dev_gyr)
ref_respiration = np.array(data.respiration.ref_respiration)

In [ ]:
plt.figure()
plt.plot(dev_lead2, label="aligned_lead2")
plt.plot(ref_lead2, label="aligned_ref_lead2")
plt.title("aligned_lead2_comparison")
plt.legend()
plt.show

plt.figure()
plt.plot(dev_ip, label="aligned_ip")
plt.plot(dev_gyr, label="aligned_gyr")
plt.plot(ref_respiration, label="aligned_ref_resp")
plt.title("aligned_respiration_comparison")
plt.legend()
plt.show

# Respiration segmentation and processing

In [ ]:
def segment_and_extract_respiration(ref_respiration):
    resp_sig_len = len(ref_respiration)
    resp_window_sec = 30
    resp_step_sec=10

    resp_W = int(resp_window_sec * fs)
    resp_step = int(resp_step_sec * fs)
    resp_segments = [(s, s + resp_W) for s in range(0, resp_sig_len - resp_W + 1, resp_step)]

    final_rr = []
    resp_ref_all_segs = []

    dev_ip_peaks_all_segs = []
    dev_gyr_peaks_all_segs = []
    dev_ref_resp_peaks_all_segs = []

    spi_ip_all_segs = []
    spi_gyr_all_segs = []
    spi_ref_all_segs = []

    for i, (s, e) in enumerate(resp_segments):
        resp_ip, ip_peaks, spi_ip = compute_resp_features_(dev_ip[s:e])
        resp_gyr, gyr_peaks, spi_gyr = compute_resp_features_(dev_gyr[s:e])
        resp_ref, ref_peaks, spi_ref = compute_resp_features_(ref_respiration[s:e])
        
        resp_ref_all_segs.append(resp_ref)

        '''
        adding segment along with the extracted peaks for diagnosis purposes;
        Not returning it for now, but just need to return following three arrays
        if needed
        '''
        dev_ip_peaks_all_segs.append([dev_ip[s:e], ip_peaks])
        dev_gyr_peaks_all_segs.append([dev_gyr[s:e], gyr_peaks])
        dev_ref_resp_peaks_all_segs.append([ref_respiration[s:e], ref_peaks])
        
        spi_ip_all_segs.append(spi_ip)
        spi_gyr_all_segs.append(spi_gyr)
        spi_ref_all_segs.append(spi_ref)

        ip_nan = np.isnan(resp_ip)
        gyr_nan = np.isnan(resp_gyr)

        if ip_nan and gyr_nan:
            final_rr.append(float('nan'))
        elif ip_nan or gyr_nan:
            final_rr.append(float(resp_gyr if ip_nan else resp_ip))
        else:
            final_rr.append(float(np.mean([resp_ip, resp_gyr])))

    resp_error = []
    for idx, rr in enumerate(final_rr):
        resp_error.append(abs(rr - resp_ref_all_segs[idx]))

    return final_rr, resp_ref_all_segs, resp_error

In [ ]:
resp_error = []
for idx, rr in enumerate(final_rr):
    resp_error.append(rr - resp_ref_all_segs[idx])

In [ ]:
rr_comp = pd.DataFrame({
    'dev_rr': final_rr,
    'ref_rr': resp_ref_all_segs,
    'error': resp_error
})
rr_comp

In [ ]:
spi_comp = pd.DataFrame({
    'spi_ip': spi_ip_all_segs,
    'spi_gyr': spi_gyr_all_segs,
    'spi_ref': spi_ref_all_segs
})
spi_comp

In [ ]:
resp_sig_len = len(pre_aligned_prep_ref_resp)
resp_window_sec = 30
resp_step_sec=10

resp_W = int(resp_window_sec * fs)
resp_step = int(resp_step_sec * fs)
resp_segments = [(s, s + resp_W) for s in range(0, resp_sig_len - resp_W + 1, resp_step)]

In [ ]:
final_rr = []
resp_ref_all_segs = []

spi_ip_all_segs = []
spi_gyr_all_segs = []
spi_ref_all_segs = []

for i, (s, e) in enumerate(resp_segments):
    resp_ip, spi_ip = compute_resp_features_(pre_aligned_prep_ip[s:e])
    resp_gyr, spi_gyr = compute_resp_features_(pre_aligned_prep_gyr[s:e])
    resp_ref, spi_ref = compute_resp_features_(pre_aligned_prep_ref_resp[s:e])
    
    resp_ref_all_segs.append(resp_ref)
    
    spi_ip_all_segs.append(spi_ip)
    spi_gyr_all_segs.append(spi_gyr)
    spi_ref_all_segs.append(spi_ref)

    ip_nan = np.isnan(resp_ip)
    gyr_nan = np.isnan(resp_gyr)

    if ip_nan and gyr_nan:
        final_rr.append(float('nan'))
    elif ip_nan or gyr_nan:
        final_rr.append(float(resp_gyr if ip_nan else resp_ip))
    else:
        final_rr.append(float(np.mean([resp_ip, resp_gyr])))

In [ ]:
resp_error = []
for idx, rr in enumerate(final_rr):
    resp_error.append(rr - resp_ref_all_segs[idx])

In [ ]:
rr_comp = pd.DataFrame({
    'dev_rr': final_rr,
    'ref_rr': resp_ref_all_segs,
    'error': resp_error
})
rr_comp

# ECG Segmentation and Processing

In [ ]:
ecg_sig_len = len(ref_lead2)
ecg_window_sec = 10

ecg_W    = int(ecg_window_sec * fs)
ecg_step = ecg_W
n = ecg_sig_len // ecg_W

ecg_segments = [(i * ecg_step, i * ecg_step + ecg_W) for i in range(n)]

In [ ]:
dev_snr_all_segs = []
dev_hr_all_segs = []
dev_rmssd_all_segs = []

ref_snr_all_segs = []
ref_hr_all_segs = []
ref_rmssd_all_segs = []

for i, (s, e) in enumerate(ecg_segments):
    dev_snr, dev_hr, dev_rmssd = compute_ecg_features(dev_lead2)
    ref_snr, ref_hr, ref_rmssd = compute_ecg_features(ref_lead2)

    dev_snr_all_segs.append(dev_snr)
    dev_hr_all_segs.append(dev_hr)
    dev_rmssd_all_segs.append(dev_rmssd)

    ref_snr_all_segs.append(ref_snr)
    ref_hr_all_segs.append(ref_hr)
    ref_rmssd_all_segs.append(ref_rmssd)

In [ ]:
ecg_error = []
for idx, hr in enumerate(dev_hr_all_segs):
    ecg_error.append(hr - ref_hr_all_segs[idx])

In [ ]:
hr_comp = pd.DataFrame({
    'dev_hr': dev_hr_all_segs,
    'ref_hr': ref_hr_all_segs,
    'error': ecg_error
})
hr_comp